# Compare baseline and pipeline outputs

This notebook matches this project's real result format: JSONL traces from `run_baseline.py` and `run_graph_pipeline.py`, with keys like `ground_truth_label`, `tutor_verdict`, `verifier_verdict`, and `final_verdict`.

It does not assume a generic `baseline_results.json` file format.

In [1]:
import json
import math
from pathlib import Path

VERDICT_TO_LABEL = {
    'correct': 'optimal',
    'suboptimal': 'valid_alternative',
    'incorrect': 'incorrect',
}

def normalize_verdict(value):
    if value is None:
        return None
    return str(value).strip().lower()

def load_records(file_path):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

    if path.suffix.lower() == '.json':
    	with path.open('r', encoding='utf-8') as f:
    		data = json.load(f)
    	if isinstance(data, list):
    		return data
    	return [data]

    records = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

def pick_id_key(records):
    if not records:
        raise ValueError('No records found.')
    for key in ['id', 'state_id', 'proof_id', 'case_id', 'index']:
        if key in records[0]:
            return key
    raise KeyError('No known ID field found. Check the JSON keys.')

def build_lookup(records):
    id_key = pick_id_key(records)
    return {r[id_key]: r for r in records if id_key in r}

def get_ground_truth(record):
    gt = record.get('ground_truth_label')
    if gt is None:
        gt = record.get('ground_truth')
    return str(gt).strip().lower() if gt is not None else None

def get_verdict(record, preferred_key=None):
    if preferred_key is not None and preferred_key in record:
        value = record.get(preferred_key)
        if value is not None:
            return normalize_verdict(value)

    for key in ['final_verdict', 'tutor_verdict', 'verifier_verdict', 'verdict']:
        if key in record:
            value = record.get(key)
            if value is not None:
                return normalize_verdict(value)
    return None

def compute_metrics(records, label='records'):
    n = len(records)
    if n == 0:
        raise ValueError(f'No records available for {label}.')

    gt_labels = [get_ground_truth(r) for r in records]
    verdicts = [get_verdict(r) for r in records]

    incorrect_idx = [i for i, g in enumerate(gt_labels) if g == 'incorrect']
    suboptimal_idx = [i for i, g in enumerate(gt_labels) if g == 'valid_alternative']

    over_val = sum(1 for i in incorrect_idx if verdicts[i] != 'incorrect') / max(len(incorrect_idx), 1)
    over_rej = sum(1 for i in suboptimal_idx if verdicts[i] == 'incorrect') / max(len(suboptimal_idx), 1)
    exact_agr = sum(1 for g, v in zip(gt_labels, verdicts) if VERDICT_TO_LABEL.get(v) == g) / n

    print(f'\n── {label} (n={n}) ──')
    print(f'  over_validation_rate:  {over_val:.4f}  ({over_val * 100:.2f}%)')
    print(f'  over_rejection_rate:   {over_rej:.4f}  ({over_rej * 100:.2f}%)')
    print(f'  exact_agreement_rate:  {exact_agr:.4f}  ({exact_agr * 100:.2f}%)')

    return {
        'over_val': over_val,
        'over_rej': over_rej,
        'exact_agr': exact_agr,
        'incorrect_idx': incorrect_idx,
        'suboptimal_idx': suboptimal_idx,
        'gt': gt_labels,
        'verdict': verdicts,
    }

def mcnemar_exact(gt_labels, verdict_a, verdict_b, label='comparison'):
    b, c = 0, 0
    for g, va, vb in zip(gt_labels, verdict_a, verdict_b):
        a_correct = VERDICT_TO_LABEL.get(va) == g
        b_correct = VERDICT_TO_LABEL.get(vb) == g
        if a_correct and not b_correct:
            b += 1
        elif b_correct and not a_correct:
            c += 1

    n_discordant = b + c
    print(f'\n── McNemar ({label}) ──')
    print(f'  b (A✓ B✗): {b}')
    print(f'  c (A✗ B✓): {c}')
    print(f'  discordant pairs: {n_discordant}')

    if n_discordant == 0:
        print('  p-value: 1.0 (no discordant pairs)')
        return 1.0

    smaller = min(b, c)
    p = 0.0
    for k in range(0, smaller + 1):
        p += math.comb(n_discordant, k) * (0.5 ** n_discordant)
    p *= 2
    p = min(p, 1.0)

    print(f'  p-value (two-tailed exact): {p:.4f}')
    if p < 0.05:
        better = 'B' if c > b else 'A'
        print(f'  ✓ Significant — {better} is significantly better (α=0.05)')
    else:
        print('  ✗ Not significant — difference is likely noise')
    return p

project_dir = Path.cwd()
if not (project_dir / 'Data' / 'llm_output').exists():
    project_dir = project_dir.parent

baseline_path = project_dir / 'Data' / 'llm_output' / 'baseline_run.jsonl'
pipeline_path = project_dir / 'Data' / 'llm_output' / 'pipeline_run.jsonl'

print('Baseline path:', baseline_path)
print('Pipeline path:', pipeline_path)

baseline_raw = load_records(baseline_path)
pipeline_raw = load_records(pipeline_path)

print('Baseline sample keys:', list(baseline_raw[0].keys())[:10])
print('Pipeline sample keys:', list(pipeline_raw[0].keys())[:10])

baseline_by_id = build_lookup(baseline_raw)
pipeline_by_id = build_lookup(pipeline_raw)
common_ids = set(baseline_by_id) & set(pipeline_by_id)

print(f'\nBaseline total: {len(baseline_by_id)}')
print(f'Pipeline total: {len(pipeline_by_id)}')
print(f'Common matched subset: {len(common_ids)}')

baseline = [baseline_by_id[i] for i in common_ids]
pipeline = [pipeline_by_id[i] for i in common_ids]

baseline_metrics = compute_metrics(baseline, 'Baseline (matched)')
pipeline_metrics = compute_metrics(pipeline, 'Pipeline (matched)')

base_verdicts = [get_verdict(r, 'tutor_verdict') for r in baseline]
pipe_verdicts = [get_verdict(r, 'final_verdict') for r in pipeline]
gt_all = [get_ground_truth(r) for r in baseline]

mcnemar_exact(gt_all, base_verdicts, pipe_verdicts, 'ALL matched steps')

incorrect_idx = [i for i, g in enumerate(gt_all) if g == 'incorrect']
gt_inc = [gt_all[i] for i in incorrect_idx]
base_inc = [base_verdicts[i] for i in incorrect_idx]
pipe_inc = [pipe_verdicts[i] for i in incorrect_idx]
mcnemar_exact(gt_inc, base_inc, pipe_inc, 'Incorrect steps only')

suboptimal_idx = [i for i, g in enumerate(gt_all) if g == 'valid_alternative']
gt_sub = [gt_all[i] for i in suboptimal_idx]
base_sub = [base_verdicts[i] for i in suboptimal_idx]
pipe_sub = [pipe_verdicts[i] for i in suboptimal_idx]
mcnemar_exact(gt_sub, base_sub, pipe_sub, 'Valid-alternative steps only')

conformity = [
    1 for r in pipeline if get_verdict(r, 'tutor_verdict') == get_verdict(r, 'verifier_verdict')
]
if conformity:
    print(f'\nConformity rate: {sum(conformity) / len(conformity):.4f}')
else:
    print('\nNo conformity entries found; check the pipeline result keys.')

Baseline path: c:\Users\chakr\Desktop\Multi-Agent-Tutoring-Systems\Multi-Agent-Tutoring-Systems\dt_code\Data\llm_output\baseline_run.jsonl
Pipeline path: c:\Users\chakr\Desktop\Multi-Agent-Tutoring-Systems\Multi-Agent-Tutoring-Systems\dt_code\Data\llm_output\pipeline_run.jsonl
Baseline sample keys: ['id', 'problem', 'givens', 'intermediates', 'conclusion', 'correct_step', 'student_next_step', 'student_rule', 'student_full_response', 'ground_truth_label']
Pipeline sample keys: ['id', 'problem', 'givens', 'intermediates', 'conclusion', 'correct_step', 'student_next_step', 'student_rule', 'student_full_response', 'ground_truth_label']

Baseline total: 509
Pipeline total: 510
Common matched subset: 503

── Baseline (matched) (n=503) ──
  over_validation_rate:  0.3758  (37.58%)
  over_rejection_rate:   0.5000  (50.00%)
  exact_agreement_rate:  0.6123  (61.23%)

── Pipeline (matched) (n=503) ──
  over_validation_rate:  0.2562  (25.62%)
  over_rejection_rate:   0.3333  (33.33%)
  exact_agreem

In [2]:
# All cases that both baseline and pipeline matched
matched_ids = sorted(common_ids)
print(f"Matched cases count: {len(matched_ids)}")
print("Matched IDs:")
for case_id in matched_ids:
    print(case_id)

matched_rows = []
for case_id in matched_ids:
    baseline_row = baseline_by_id[case_id]
    pipeline_row = pipeline_by_id[case_id]
    matched_rows.append({
        'id': case_id,
        'problem': baseline_row.get('problem') or baseline_row.get('currentProblem'),
        'ground_truth': get_ground_truth(baseline_row),
        'baseline_verdict': get_verdict(baseline_row, 'tutor_verdict'),
        'pipeline_verdict': get_verdict(pipeline_row, 'final_verdict'),
        'pipeline_tutor_verdict': get_verdict(pipeline_row, 'tutor_verdict'),
        'pipeline_verifier_verdict': get_verdict(pipeline_row, 'verifier_verdict'),
    })

matched_rows

Matched cases count: 503
Matched IDs:
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
66
67
68
69
70
71
72
74
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
98
99
100
101
102
103
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
2

[{'id': 1,
  'problem': '2.2',
  'ground_truth': 'incorrect',
  'baseline_verdict': 'incorrect',
  'pipeline_verdict': 'incorrect',
  'pipeline_tutor_verdict': 'incorrect',
  'pipeline_verifier_verdict': 'incorrect'},
 {'id': 2,
  'problem': '2.2',
  'ground_truth': 'incorrect',
  'baseline_verdict': 'incorrect',
  'pipeline_verdict': 'incorrect',
  'pipeline_tutor_verdict': 'incorrect',
  'pipeline_verifier_verdict': 'incorrect'},
 {'id': 3,
  'problem': '2.3',
  'ground_truth': 'incorrect',
  'baseline_verdict': 'correct',
  'pipeline_verdict': 'incorrect',
  'pipeline_tutor_verdict': 'incorrect',
  'pipeline_verifier_verdict': 'incorrect'},
 {'id': 4,
  'problem': '2.3',
  'ground_truth': 'incorrect',
  'baseline_verdict': 'correct',
  'pipeline_verdict': 'incorrect',
  'pipeline_tutor_verdict': 'incorrect',
  'pipeline_verifier_verdict': 'incorrect'},
 {'id': 5,
  'problem': '2.3',
  'ground_truth': 'incorrect',
  'baseline_verdict': 'incorrect',
  'pipeline_verdict': 'incorrect',


In [7]:
from collections import Counter

gt_dist = Counter(t["ground_truth_label"] for t in baseline)  # same ids as pipeline
total = sum(gt_dist.values())

for label, count in gt_dist.most_common():
    print(f"{label:20s} {count:4d}  ({count/total:.1%})")

incorrect             479  (95.2%)
optimal                16  (3.2%)
valid_alternative       8  (1.6%)
